## 数据集划分

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option('display.unicode.east_asian_width', True) # 优化中文对齐

df_encode = pd.read_excel("../data/Telco_customer_churn_final_model.xlsx")
df_encode.head()

,老年人,伴侣,家属,在网时长（月）,互联网服务,网络安全,技术支持,流媒体电视,流媒体电影,合同类型,无纸化账单,支付方式,月费用,总费用,客户终身价值,流失值
0,0,0,0,-1.236724,1,1,0,0,0,0.083,1,0,-0.362660,-0.958066,-0.981675,1
1,0,0,1,-1.236724,2,0,0,0,0,0.083,1,0,0.197365,-0.938874,-1.436462,1
2,0,0,1,-0.992402,2,0,0,1,1,0.083,1,0,1.159546,-0.643789,0.821409,1
3,0,1,1,-0.177995,2,0,1,1,1,0.083,1,0,1.330711,0.338085,0.509483,1
4,0,0,1,0.677133,2,0,0,1,1,0.083,1,1,1.294151,1.216150,0.794358,1


In [2]:
# %pip install imblearn

In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

current_df = df_encode 

X = current_df.drop('流失值', axis=1)
y = current_df['流失值']

# --- 第一刀：切出 10% 测试集 (Test) ---
# 剩余 90% 留作 (Train + Val)
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, 
    test_size=0.1, 
    random_state=42, 
    stratify=y  # 关键：分层采样，保持流失率一致
)

# --- 第二刀：切出 20% 验证集 (Val) ---
# 验证集占总数的 0.2，即占剩余部分的 0.2 / 0.9 = 2/9
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, 
    test_size=2/9, 
    random_state=42, 
    stratify=y_temp 
)

print(f"原始样本数: {len(X)}")
print(f"训练集 (Train): {len(X_train)} ({len(X_train)/len(X):.1%})")
print(f"验证集 (Val):   {len(X_val)}   ({len(X_val)/len(X):.1%})")
print(f"测试集 (Test):  {len(X_test)}   ({len(X_test)/len(X):.1%})")


# ==========================================
# 2. 定义统计量辅助函数 (Helper Function)
# ==========================================
def get_stats_raw(df, tag):
    """
    计算指定数据集的均值和标准差，并打上标签
    """
    # 仅计算数值型特征，防止报错
    stats = df.describe().loc[['mean', 'std']]
    stats['Dataset'] = tag
    stats['Metric'] = stats.index 
    return stats

# 定义需要检查分布一致性的关键数值特征
check_feats = ['在网时长（月）', '月费用', '客户终身价值', '总费用']


# ==========================================
# 3. 表1：各数据集分布一致性检验 (Original vs Train vs Val vs Test)
# ==========================================
# 获取各部分的统计量
s_orig = get_stats_raw(X[check_feats], 'Original')
s_train = get_stats_raw(X_train[check_feats], 'Train')
s_val = get_stats_raw(X_val[check_feats], 'Val')
s_test = get_stats_raw(X_test[check_feats], 'Test')

# 合并并透视
all_stats = pd.concat([s_orig, s_train, s_val, s_test])
df_melt = all_stats.melt(id_vars=['Dataset', 'Metric'], var_name='Feature', value_name='Value')

dist_table = df_melt.pivot_table(
    index='Feature', 
    columns=['Dataset', 'Metric'], 
    values='Value'
)

# 调整列顺序
desired_order = ['Original', 'Train', 'Val', 'Test']
dist_table = dist_table.reindex(columns=desired_order, level=0)

print("\n" + "="*60)
print("Table 1: 数据集划分分布一致性检验 (Distribution Consistency Check)")
print("="*60)
dist_table.round(4)

原始样本数: 7043
训练集 (Train): 4929 (70.0%)
验证集 (Val):   1409   (20.0%)
测试集 (Test):  705   (10.0%)

Table 1: 数据集划分分布一致性检验 (Distribution Consistency Check)


Dataset        Original           Train             Val            Test  \
Metric             mean     std    mean     std    mean     std    mean   
Feature                                                                   
在网时长（月）     -0.0  1.0001 -0.0048  1.0024 -0.0053  0.9908  0.0439   
客户终身价值       -0.0  1.0001  0.0018  0.9977 -0.0069  1.0090  0.0010   
总费用             -0.0  1.0001 -0.0056  0.9974  0.0094  1.0134  0.0208   
月费用             -0.0  1.0001 -0.0052  0.9991  0.0173  1.0132  0.0021   

Dataset                 
Metric             std  
Feature                 
在网时长（月）  1.0026  
客户终身价值    1.0000  
总费用          0.9931  
月费用          0.9813

In [6]:
# ==========================================
# 4. 类别不平衡处理 (SMOTE on Train Set Only)
# ==========================================
print("\n" + "="*60)
print("正在对训练集进行 SMOTE 过采样 (Over-sampling)...")
print("="*60)

print(f"SMOTE 前训练集形状: {X_train.shape}")
print(f"SMOTE 前流失分布: \n{y_train.value_counts(normalize=True)}")

# 初始化并训练 SMOTE
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("-" * 30)
print(f"SMOTE 后训练集形状: {X_train_smote.shape}")
print(f"SMOTE 后流失分布: \n{y_train_smote.value_counts(normalize=True)}")


# ==========================================
# 5. 表2：SMOTE 前后数据质量检验 (Train vs Train_SMOTE)
# ==========================================
# 获取 SMOTE 后的统计量
s_train_smote = get_stats_raw(X_train_smote[check_feats], 'Train_SMOTE')

# 对比：原始训练集 (s_train) vs SMOTE后训练集 (s_train_smote)
smote_comparison = pd.concat([s_train, s_train_smote])

# 透视处理
smote_melt = smote_comparison.melt(id_vars=['Dataset', 'Metric'], var_name='Feature', value_name='Value')
smote_table = smote_melt.pivot_table(
    index='Feature', 
    columns=['Dataset', 'Metric'], 
    values='Value'
)

# 调整列顺序 (只对比 Train 和 Train_SMOTE)
smote_order = ['Train', 'Train_SMOTE']
smote_table = smote_table.reindex(columns=smote_order, level=0)

print("\n" + "="*60)
print("Table 2: SMOTE 数据生成质量检验 (SMOTE Quality Check)")
print("="*60)
smote_table.round(4)


正在对训练集进行 SMOTE 过采样 (Over-sampling)...
SMOTE 前训练集形状: (4929, 15)
SMOTE 前流失分布: 
流失值
0    0.734632
1    0.265368
Name: proportion, dtype: float64
------------------------------
SMOTE 后训练集形状: (7242, 15)
SMOTE 后流失分布: 
流失值
0    0.5
1    0.5
Name: proportion, dtype: float64

Table 2: SMOTE 数据生成质量检验 (SMOTE Quality Check)


Dataset          Train         Train_SMOTE        
Metric            mean     std        mean     std
Feature                                           
在网时长（月） -0.0048  1.0024     -0.1998  0.9817
客户终身价值    0.0018  0.9977     -0.0685  0.9958
总费用         -0.0056  0.9974     -0.1191  0.9604
月费用         -0.0052  0.9991      0.0913  0.9549

In [8]:
import pickle
import os

save_dir = "../data/"
if not os.path.exists(save_dir):
    os.makedirs(save_dir)

data_bundle = {
    # --- 数据集 ---
    'X_train': X_train,         'y_train': y_train,
    'X_train_smote': X_train_smote, 'y_train_smote': y_train_smote,
    'X_val': X_val,             'y_val': y_val,
    'X_test': X_test,           'y_test': y_test,
    
    # --- 元数据 ---
    'feature_names': X_train.columns.tolist(), # 特征名称列表
    
    # --- 原始 CLTV (用于代价敏感加权) ---
    # 如果 X_train 里已经是标准化后的 CLTV，我们可能需要反归一化，或者直接用相对大小
    # 这里我们直接保存 X_train 里的 CLTV 列即可，后续在模型里取出来做权重
    'cltv_col_name': '客户终身价值' 
}

# 3. 保存为 Pickle 文件
file_path = os.path.join(save_dir, "modeling_data_ready.pkl")
with open(file_path, 'wb') as f:
    pickle.dump(data_bundle, f)

print(f"✅ 数据已打包保存至: {file_path}")
print("包含键值:", list(data_bundle.keys()))

✅ 数据已打包保存至: ../data/modeling_data_ready.pkl
包含键值: ['X_train', 'y_train', 'X_train_smote', 'y_train_smote', 'X_val', 'y_val', 'X_test', 'y_test', 'feature_names', 'cltv_col_name']
